# Send-side quota enforcement in one-time-password software

**A reproduction notebook for the package survey and the concurrency measurement.
Artifact version `v5.0-r17`.**

An SMS one-time-password endpoint spends money every time it accepts a request. This
notebook reruns the SQLite main sweep, regenerates the reported analyses from the archived
request-level records, and provides commands for rerunning the survey and the complete
round-4 experiment suite. It does not itself rerun the survey or every Study 2 arm. The two
measurements behind the paper are:

1. **Study 1, the survey.** Of 42 sampled packages on the Python Package Index whose
   purpose is delivering a one-time or verification code, how many place any limit on
   *sending*, how many on *verifying*, and how many on both?
2. **Study 2, the measurement.** When the application writes the limit itself, does the
   limit survive requests that arrive at the same time, and which request interleavings
   produce which overruns?

The notebook runs in four parts. Part A shows the defect in a single cell with no server
at all. Part B starts the real service (FastAPI, SQLite, four worker processes), runs the
paper's harness against it over HTTP, keeps one event record per request, and evaluates
each implementation against the policy it actually enforces on its admission events.
Part C runs `regenerate.py`, the single script that recomputes every server-side Study 2
count and policy classification in the paper from the event records shipped in
`results/r4/` (and the survey's numbers from its frozen per-package audit), and displays
the tables and figures.
Part D reads the per-overrun classification and checks one overrun's window assignment
by hand from its event records.

**Nothing here sends a real message.** The provider is a local stub that sleeps for a
configurable interval and records the attempt. No telephone number in this notebook is
real, no external API is contacted, no package maintainer is contacted, and no production
system is involved.

| | |
|---|---|
| Part A | in-process, about ten seconds |
| Part B | the real service, SQLite main sweep only, at the paper's trial counts: about 15 minutes on two cores (`REDUCED = True` for about two) |
| Part C | `regenerate.py` over the shipped event records and the frozen survey audit, under a minute; no arm is re-run |
| Part D | reads `results/overrun_classification.csv` and one event record, instant |
| Other Study 2 arms | not run here; `reproduce.sh` re-runs all of them (about two hours) |
| Study 1 re-run | not run here; about 20 minutes and network access; commands at the end |

Table and figure numbers below follow manuscript v4.8 and its supplement (S numbers).

## 0. Setup

Run this notebook from inside a checkout of the artifact. It needs `fastapi`, `uvicorn`,
`httpx`, `pandas` and `matplotlib`; `requirements-lock.txt` pins the versions the reported
runs used.

In [ ]:
import importlib.util
import inspect
import json
import os
import pathlib
import statistics
import sys
import tempfile
import threading
import time

ROOT = pathlib.Path.cwd()
while not (ROOT / "svc" / "app.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "svc" / "app.py").exists(), (
    "Could not find svc/app.py. Run this notebook from inside the artifact repository."
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

RESULTS = ROOT / "results"
missing = [m for m in ("fastapi", "uvicorn", "httpx", "pandas", "matplotlib")
           if importlib.util.find_spec(m) is None]

print("artifact version:", (ROOT / "VERSION").read_text().strip())
print("repository root :", ROOT)
print("python          :", sys.version.split()[0])
print("missing packages:", ", ".join(missing) if missing else "none")
print("shipped results :", len(list(RESULTS.glob("**/*.csv"))), "csv files,",
      len(list(RESULTS.glob("**/*.json"))), "json files")

---
## 1. The six implementations of one policy

The declared policy throughout is the same: **at most three sends per phone number per
window.** Three implementations count records in a trailing (sliding) window of 3600
seconds; three keep a counter that resets at aligned boundaries (a fixed window). The two
are different policies, and the harness judges each implementation against its own.

Four of the six are structural analogues of code found in the packages surveyed in Study 1
(Table II in the paper states exactly what each source does and where the fixture differs);
two are remedies no sampled package uses. **The policy event** is the admission: the
completion of the implementation's own granting state update. For the two one-step
fixed-window implementations the database returns the window the admission counts in from
the granting statement itself (`RETURNING window_start, used`); for the read-modify-write
counter the recorded window is the one the request computed and wrote, because that pattern
decides in the application (`window_source` in each record says which); for the
sliding-window ones the admission timestamp is the value the implementation used in its own
check. The provider call comes
after the admission and its instant is never used to assign a window. Three do their bookkeeping in two steps, reading
the state and then writing it. Three decide in one step that the datastore serialises.

| Pattern | Steps | Policy | What the code does |
|---|---|---|---|
| `naive_count` | two | sliding | Count recent send records, compare against the limit, insert a record and send |
| `cache_get_set` | two | fixed | Read a counter, add one, write it back: the shape most cache-backed limiters take |
| `after_send` | two | sliding | Check the count, send the message, and only then record the send |
| `atomic_update` | one | fixed | Increment the counter only if it is below the limit, in one statement, and let the datastore report whether that happened |
| `immediate_txn` | one | sliding | Check and increment inside a single write transaction |
| `reserve_commit` | one | fixed | Claim the allowance with the atomic statement before calling the provider, and release it if the call fails |

`after_send` deserves a note, because it is the most *attractive* of the three two-step
ones. Incrementing after a successful send means a failed send does not consume the user's
allowance, which is a real and reasonable goal. The instinct is correct; the fix is to
release a reservation on failure, not to defer the claim.

Here is the source of all six, read directly from the service the experiments run against.

In [ ]:
os.environ.update(
    OTP_DB=str(pathlib.Path(tempfile.mkdtemp(prefix="otpnb-")) / "demo.db"),
    OTP_LIMIT="3",
    OTP_WINDOW_S="3600",
    OTP_PROVIDER_MS="50",
    OTP_STRATEGY="naive_count",
)

import svc.app as S  # noqa: E402  (env must be set before import)

ORDER = ["naive_count", "cache_get_set", "after_send",
         "atomic_update", "immediate_txn", "reserve_commit"]
LABEL = {
    "naive_count": "Count-then-insert",
    "cache_get_set": "Read-modify-write counter",
    "after_send": "Increment after send",
    "atomic_update": "Conditional atomic update",
    "immediate_txn": "Guarded write transaction",
    "reserve_commit": "Reserve before send",
}

for name in ORDER:
    print("=" * 74)
    print(f"{name}   ({LABEL[name]})")
    print("=" * 74)
    print(inspect.getsource(S.STRATEGIES[name]))

And the provider stub, which is the only thing in the service that would cost money in the
real world. Every request also writes one event record (`record_event`) after it finishes;
that record, not the provider row, is what the policy is evaluated on.

In [ ]:
print(inspect.getsource(S.provider_send))
print(inspect.getsource(S.record_event))
print("declared limit :", S.LIMIT)
print("window         :", S.WINDOW_S, "s")
print("provider stub  :", S.PROVIDER_MS, "ms")

---
## Part A: the race, in one cell

No server, no HTTP, no worker processes. We call the six quota functions directly from
threads that are released at the same instant by a barrier, and then count how many calls
reached the provider stub.

This is the shortest honest demonstration of the defect. It takes about ten seconds.

In [ ]:
S.init_db()


def reset():
    c = S.conn()
    for t in ("attempts", "counters", "provider_sends", "reservations"):
        c.execute(f"DELETE FROM {t}")
    c.close()


def provider_calls(phone):
    c = S.conn()
    n = c.execute("SELECT COUNT(*) FROM provider_sends WHERE phone=?",
                  (phone,)).fetchone()[0]
    c.close()
    return n


ERRORS = []


def burst_inprocess(fn, phone, k):
    # Release k calls to fn at the same instant; return recorded provider calls.
    # Exceptions are recorded, never swallowed: a failed client observation is
    # not the same thing as no provider action.
    gate = threading.Barrier(k)

    def worker():
        gate.wait()
        try:
            fn(phone)
        except Exception as e:
            ERRORS.append(type(e).__name__)

    threads = [threading.Thread(target=worker) for _ in range(k)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return provider_calls(phone)


K, TRIALS = 8, 5
print(f"declared limit = {S.LIMIT}    simultaneous requests = {K}    trials = {TRIALS}\n")
print(f"{'pattern':<28}{'provider calls per trial':<30}{'worst'}")
print("-" * 68)
for name in ORDER:
    got = []
    for t in range(TRIALS):
        phone = f"+1000{name[:4]}{t}"
        reset()
        got.append(burst_inprocess(S.STRATEGIES[name], phone, K))
    print(f"{LABEL[name]:<28}{str(got):<30}{max(got)}")
print(f"\nexceptions raised across all calls: {len(ERRORS)}"
      + (f"  {sorted(set(ERRORS))}" if ERRORS else ""))

The three two-step patterns overshoot; the three one-step patterns do not. Every one of
those calls returned normally: the two-step implementations did not fail, they *succeeded*
more times than the policy allows.

Part A is deliberately artificial in one respect: it calls the quota functions in-process,
which is not how anybody deploys an OTP endpoint. Part B removes that objection.

---
## Part B: the real service, four worker processes, over HTTP

This is the configuration the paper measures: `uvicorn` with `--workers 4` in front of the
FastAPI app, SQLite in write-ahead-logging mode behind it, and requests arriving as real
HTTP POSTs from a thread barrier in a separate process.

Nothing in the service is instrumented to widen a race. There is no injected sleep inside a
critical section and no lock removed for effect. The only tunable is the provider stub's
latency, and Part C shows the result does not depend on it.

The harness is `study/run_experiments.py`. After every burst it fetches one event record
per request from the service (`GET /otp/events`) and evaluates the declared policy on the
admission events with `study/policy.py`: the largest number of admissions in any window
identifier for the fixed-window implementations, the largest number in any trailing
interval for the sliding-window ones. Both the raw total and the policy-relevant maximum
are kept, because a burst that straddles a fixed-window boundary can legitimately exceed
the limit in total while exceeding it in no window. That distinction is the subject of
Part D.

In [ ]:
os.environ["EXP_PORT"] = "8131"
os.environ["EXP_DB"] = str(pathlib.Path(tempfile.mkdtemp(prefix="otpnb-")) / "exp.db")
os.environ["EXP_EVENT_DIR"] = os.environ["EXP_DB"] + ".events"

import study.run_experiments as RC  # noqa: E402
import study.policy as POL  # noqa: E402

print("harness endpoint:", RC.BASE)
print("policy per implementation:", RC.POLICY)
print()
print(inspect.getsource(RC.burst))
print(inspect.getsource(POL.evaluate))

### Running the sweep

`REDUCED = True` runs 3 trials per cell, which takes about two minutes and is enough to see
the effect. The default, `False`, runs the paper's 30 trials per cell: 1 080 trials, about
15 minutes on two cores. Every request's event record is kept in `events`.

In [ ]:
# Full reproduction is the default: this is the paper's trial count.
# Set REDUCED = True for a two-minute run that shows the effect but does not
# reproduce the published figures.
REDUCED = False
TRIALS = 3 if REDUCED else 30
KS = [1, 2, 4, 8, 16, 32]

rows, events = [], []
t0 = time.time()
for name in ORDER:
    proc = RC.start_server(name, 3, 50.0, "sqlite", 4)
    try:
        for k in KS:
            for trial in range(TRIALS):
                phone = f"+1999{name[:4]}{k}{trial}"
                RC.httpx.post(f"{RC.BASE}/admin/reset", timeout=30.0)
                replies = RC.burst(phone, k)
                evs, lim, win, pol = RC.events_for(phone)
                ev = POL.evaluate(evs, lim, win, pol)
                for e in evs:
                    # the service stamps its own strategy name on every
                    # record; the notebook's identity fields win, as in the
                    # harness
                    events.append({**e, "strategy": name, "k": k, "trial": trial})
                rows.append(dict(
                    strategy=name, k=k, trial=trial,
                    admits=ev["admits"], provider_calls=ev["provider_calls"],
                    requests_logged=ev["requests"], max_in_window=ev["max_per_policy"],
                    policy_overrun=ev["policy_overrun"], windows=ev["windows_touched"],
                    accepted=sum(1 for r in replies if r == "sent"),
                    rejected=sum(1 for r in replies if r == "rejected"),
                    errors=sum(1 for r in replies if str(r).startswith("error")),
                ))
    finally:
        RC.stop_server(proc)
    print(f"  {LABEL[name]:<28} done   ({time.time() - t0:5.0f}s elapsed)")

print(f"\n{len(rows)} trials, {len(events)} event records, in {time.time() - t0:.0f}s; "
      f"client errors: {sum(r['errors'] for r in rows)}; "
      f"trials where provider calls != admissions: {sum(r['provider_calls'] != r['admits'] for r in rows)}")

In [ ]:
import pandas as pd  # noqa: E402

run = pd.DataFrame(rows)


def sweep_table(df, kcol="k", vcol="admits", limit=3):
    out = []
    for s in ORDER:
        d = df[df.strategy == s]
        r = {"pattern": LABEL[s]}
        for k in sorted(df[kcol].unique()):
            v = d[d[kcol] == k][vcol]
            r[f"k={k}"] = f"{v.median():g}/{int(v.max())}"
        r["over (total)"] = int((d[vcol] > limit).sum())
        r["policy violations"] = int((d["policy_overrun"] > 0).sum())
        r["worst"] = f"{d[vcol].max() / limit:.1f}x"
        out.append(r)
    return pd.DataFrame(out).set_index("pattern")


print("admissions against a declared limit of 3: median / maximum per cell\n")
sweep_table(run)

Two things to read out of that table.

At `k=1` and `k=2` every implementation is correct. The defect is invisible to any test
that does not overlap requests, which is most test suites, because issuing requests in a
loop and asserting that the fourth is rejected is the obvious way to test a rate limit, and
it passes against all six.

Further up the range, the two-step implementations stop being merely wrong and start
matching the request count: whatever the attacker sends, the service delivers. The reply to
every one of those requests was a success. The two "over" columns agree here because a
one-hour window is almost never straddled by a burst; Part D shows what happens when it is.

In [ ]:
# one event record, so the fields are visible
print(json.dumps(next(e for e in events if e["decision"] == "admit" and e["strategy"] == "atomic_update"), indent=1))

---
## Part C: every number in the paper, regenerated from the event records

`regenerate.py` reads the event records in `results/r4/` and the per-package audit, evaluates
the policy on the admission events, derives every per-trial summary and checks it against the
harness's own CSV (it stops if they disagree), and recomputes every count, median, maximum,
quartile, exceedance count, Wilson interval and cross-tabulation in the paper into
`results/paper_numbers.json`. It also writes `results/overrun_classification.csv`. (The
SHA-256 manifest is written separately by `study/manifest.py`, last, when the archive is
assembled.) `study/paper_tables.py` then writes the LaTeX bodies of the result tables
from that file, so no measured number in a result table is typed by hand.

In [ ]:
import regenerate as RG  # noqa: E402

RG.main([])   # [] : do not parse the kernel's own command line
P = json.load(open(RESULTS / "paper_numbers.json"))
S1, S2, OV, H = P["study1"], P["study2"], P["overruns"], P["historical"]
print("\nartifact version:", P["version"])
print("round-4 arms:", S2["arms"], "total", S2["arms_total"])
print("historical arms total:", H["arms_total"])

### Table III of the paper and Table S5 of the supplement: every experimental arm

In [ ]:
arms = pd.Series(S2["arms"], name="round-4 trials")
arms.loc["total"] = S2["arms_total"]
display(arms.to_frame())
hist = pd.Series(H["arms"], name="historical trials (rounds 1-3)")
hist.loc["total"] = H["arms_total"]
hist.to_frame()

### Table VI: admissions under concurrent load (SQLite main sweep)

Each cell is median / maximum over 30 trials with the number of trials exceeding the limit in
total; "PV" is the number whose declared policy was violated per window.

In [ ]:
def paper_sweep(name):
    ks = S2["concurrency_levels"]
    out = []
    for s in ORDER:
        r = {"pattern": LABEL[s]}
        for k in ks:
            c = S2[name][f"{s}|{k}"]
            r[f"k={k}"] = f"{c['median']:g}/{c['max']} ({c['over_total']})"
        tot = S2[name + "_summary"][s]
        r["over"] = tot["over_total"]
        r["PV"] = tot["policy_violations"]
        r["worst"] = f"{tot['worst_amplification']}x"
        out.append(r)
    return pd.DataFrame(out).set_index("pattern")


paper_sweep("main_sqlite")

### Table S8 (supplement): quartiles of the cells whose trials are not all alike, with Wilson intervals

In [ ]:
out = []
for name, ds in (("main_sqlite", "SQLite"), ("main_postgres", "PostgreSQL")):
    for s in ORDER:
        for k in S2["concurrency_levels"]:
            c = S2[name][f"{s}|{k}"]
            if c["min"] != c["max"]:
                out.append(dict(datastore=ds, pattern=LABEL[s], k=k, min=c["min"],
                                Q1=c["q1"], median=c["median"], Q3=c["q3"], max=c["max"],
                                over=c["over_total"], over_ci_pct=c["over_total_ci_pct"]))
pd.DataFrame(out)

### Figure S4 (supplement): median admissions against simultaneous requests

The three one-step implementations have identical medians at every level, min(k, 3), and are
drawn on the same points with markers of different sizes so that each remains visible.

In [ ]:
import matplotlib.pyplot as plt  # noqa: E402

LIMIT = S2["declared_limit"]
ks = S2["concurrency_levels"]
med = {s: [S2["main_sqlite"][f"{s}|{k}"]["median"] for k in ks] for s in ORDER}

fig, ax = plt.subplots(figsize=(6.2, 3.8))
styles = [("after_send", "-", "o", 5, "Increment after send"),
          ("cache_get_set", "-.", "s", 5, "Read-modify-write counter"),
          ("naive_count", ":", "^", 6, "Count-then-insert"),
          ("immediate_txn", "-", "o", 10, "Guarded write transaction"),
          ("atomic_update", "-", "D", 5, "Conditional atomic update"),
          ("reserve_commit", "-.", "x", 6, "Reserve before send")]
for key, ls, mk, ms, lab in styles:
    kw = dict(markerfacecolor="none") if mk == "o" and ms == 10 else {}
    ax.plot(range(len(ks)), med[key], ls, marker=mk, label=lab, linewidth=1.4,
            markersize=ms, **kw)
ax.axhline(LIMIT, color="black", linestyle="--", linewidth=1)
ax.annotate("declared limit", xy=(0.15, LIMIT), xytext=(0.15, LIMIT + 1.4), fontsize=9)
ax.set_xticks(range(len(ks)))
ax.set_xticklabels(ks)
ax.set_yticks([0, LIMIT, 8, 16, 32])
ax.set_xlabel("Simultaneous requests")
ax.set_ylabel("Admissions (median)")
ax.legend(fontsize=8, loc="upper left")
ax.margins(x=0.04)
fig.tight_layout()
plt.show()

expected = [min(k, LIMIT) for k in ks]
for s in ("atomic_update", "immediate_txn", "reserve_commit"):
    assert med[s] == expected, f"{s} did not hold the declared limit"
print("checked: the three one-step patterns are identical at every level, and each admits")
print("        min(requests, declared limit), never more, in the main sweep.")

### Figure S5 (supplement): the same requests, overlapped and sequential

In [ ]:
con32 = {s: S2["main_sqlite"][f"{s}|32"]["median"] for s in ORDER}
seq32 = {s: S2["sequential"][f"{s}|32"]["median"] for s in ORDER}

rowlab, a, b = [], [], []
for s in reversed(ORDER):
    rowlab.append(LABEL[s])
    a.append(con32[s])
    b.append(seq32[s])

y = range(len(rowlab))
fig, ax = plt.subplots(figsize=(6.2, 3.2))
ax.barh([i + 0.19 for i in y], a, height=0.36, hatch="///",
        facecolor="white", edgecolor="black", label="Released simultaneously")
ax.barh([i - 0.19 for i in y], b, height=0.36,
        facecolor="white", edgecolor="black", label="Released one at a time")
for i, (va, vb) in enumerate(zip(a, b)):
    ax.text(va + 0.6, i + 0.19, f"{va:g}", va="center", fontsize=9)
    ax.text(vb + 0.6, i - 0.19, f"{vb:g}", va="center", fontsize=9)
ax.set_yticks(list(y))
ax.set_yticklabels(rowlab, fontsize=9)
ax.set_xticks([0, 8, 16, 24, 32])
ax.set_xlabel("Admissions from 32 requests (median)")
ax.legend(fontsize=9, loc="lower right")
fig.tight_layout()
plt.show()

print("sequential control, every cell exactly at the limit:", bool(S2["sequential_all_equal_limit"]))

### Table S9 (supplement): is the race just the slow provider call?

In [ ]:
out = []
for s in ORDER:
    r = {"pattern": LABEL[s]}
    for ms in (0.0, 10.0, 50.0, 200.0):
        c = S2["latency"][f"{s}|{ms}"]
        r[f"{ms:g} ms"] = f"{c['median']:g} ({c['over_total']}/{c['n']})"
    out.append(r)
print("median admissions at 8 simultaneous requests "
      "(trials exceeding the declared limit of 3 in brackets)\n")
pd.DataFrame(out).set_index("pattern")

### Table VII: the same sweep on PostgreSQL 16, and the pooled arm

In [ ]:
print("PostgreSQL 16, one connection per request:")
display(paper_sweep("main_postgres"))
print("\nPostgreSQL 16, pool of eight (k = 8 and 32):")
pd.DataFrame({k: {kk: v[kk] for kk in ("median", "max", "over_total", "policy_violations")}
              for k, v in S2["postgres_pooled"].items()}).T

### Sensitivity arms: workers and declared limit (Section V-G)

In [ ]:
print("worker count (strategy|workers|k):")
display(pd.DataFrame({k: {kk: v[kk] for kk in ("median", "max", "over_total", "policy_violations")}
                      for k, v in S2["workers"].items()}).T)
print("\ndeclared limit at k=16 (strategy|limit):")
pd.DataFrame({k: {kk: v[kk] for kk in ("median", "max", "over_total", "policy_violations")}
              for k, v in S2["limits"].items()}).T

### Fig. 6 and Table S10: a surveyed package executed as shipped

`study/run_package_exec.py` drives `django-otp` 1.7.3's own `EmailDevice.generate_challenge`
through the same barrier protocol, with a counting e-mail backend that opens no connection,
and records per-call events: the instant each thread loaded its instance, the cooldown
timestamp that instance carried, and the instants of the `send_mail` calls it made.

In [ ]:
out = []
for mode in ("sequential", "concurrent"):
    r = {"mode": mode}
    for k in (1, 2, 4, 8, 16, 32):
        c = S2["package_exec"][f"{mode}|k={k}"]
        r[f"k={k}"] = f"{c['median']:g}/{c['max']} ({c['over_limit']})"
    out.append(r)
print(f"deliveries attempted, median/max (trials over the limit of 1); "
      f"concurrent over, all k: {S2['package_exec_concurrent_over_all_k']} of "
      f"{S2['package_exec_concurrent_trials_all_k']}; k>=2: "
      f"{S2['package_exec_concurrent_over_k_ge_2']} of "
      f"{S2['package_exec_concurrent_trials_k_ge_2']}; errors {S2['package_exec_errors']}; "
      f"excess admissions {S2['package_exec_excess_admissions']}, of which loaded a null cooldown: "
      f"{S2['package_exec_excess_admissions_loaded_null_cooldown']}\n")
pd.DataFrame(out).set_index("mode")

### Table S11 (supplement): the race window, measured (admitted requests only)

The window is the time from reading the quota state to writing it back, over admitted
requests. For the conditional atomic update the same two instants bracket the single
granting statement, so that column is the statement's execution time and not a race window;
the execution time of refused statements is reported separately. All instants are
application-observed.

In [ ]:
out = []
for key, ds in (("main_sqlite", "SQLite"), ("main_postgres", "PostgreSQL"), ("postgres_pooled", "PostgreSQL, pooled")):
    for k, v in S2[key + "_timing"].items():
        s, kk = k.split("|")
        if int(kk) in (2, 8, 32):
            out.append(dict(datastore=ds, pattern=LABEL[s], k=int(kk),
                            admitted_window_ms_median=v["admitted_read_to_write_ms_median"],
                            admitted_window_ms_p90=v["admitted_read_to_write_ms_p90"],
                            refused_statement_ms_median=v["refused_statement_ms_median"],
                            spread_ms_median=v["arrival_spread_ms_median"],
                            spread_ms_max=v["arrival_spread_ms_max"]))
pd.DataFrame(out)

### Table VIII (and Table S12 in full): which interleavings produced which overruns

Primary evidence is database-visible state: the value each request's own read returned
(`observed`) and the counter value each admitted request wrote (`counter_after`). An admission
beyond the limit that observed a value below the limit read stale state; admissions minus
the largest counter value ever written is a lower bound on lost updates. The
application-observed instant comparisons are proxies for the database's order.

In [ ]:
out = []
for arm, ds in (("main_sqlite", "SQLite"), ("main_postgres", "PostgreSQL"), ("postgres_pooled", "PostgreSQL, pooled")):
    for key, x in S2[arm + "_interleaving"].items():
        s, kk = key.split("|")
        if int(kk) in (8, 32):
            out.append(dict(datastore=ds, pattern=LABEL[s], k=int(kk),
                            overrun_trials=x["overrun_trials"], excess=x["excess_admissions"],
                            excess_observed_below_limit=x["excess_observed_below_limit"],
                            lost_updates_lower_bound=x.get("lost_updates_lower_bound", ""),
                            trials_with_lost_update=x.get("trials_with_lost_update", ""),
                            excess_read_before_T3_app=x.get("excess_read_before_Lth_write_app", ""),
                            refused_issued_before_T3_app=x.get("refused_issued_before_Lth_write_app", ""),
                            refused=x["refused"],
                            first_arrival_to_T3_ms=x.get("first_arrival_to_Lth_write_ms_median", ""),
                            arrival_spread_ms=x.get("arrival_spread_ms_median", ""),
                            read_spread_ms=x.get("read_spread_ms_median", "")))
pd.DataFrame(out)

### Table S13 (supplement): the deterministic window boundary test, both datastores

A four-second fixed window, a limit of five, eight requests released one second before an
aligned boundary and eight 50 ms after it. Every admission's window is the one its own
granting statement counted it in: returned by the database for the conditional atomic update
and the reservation, written by the request for the read-modify-write counter.

In [ ]:
out = []
for arm in ("boundary_sqlite", "boundary_postgres"):
    for s in ORDER:
        b = S2[arm][s]
        out.append(dict(arm=arm, implementation=LABEL[s], policy=b["policy"], trials=b["trials"],
                        straddled=b["straddled"], total_over_5=b["total_exceeds_limit"],
                        max_per_window=b["max_per_policy"], violated=b["policy_violated"],
                        admits_before_after=b["admits_before_after"]))
pd.DataFrame(out)

### Table IX, Figs. 7 and 8, and Table S14: the formal models (archived TLC and Tamarin results)

The models in `formal/` (TLA+ checked with TLC, and Tamarin) are not rerun here; TLC needs
Java and Tamarin needs Maude, and `formal/README.md` gives the commands and the pinned
versions. This cell reads the archived verdicts that `regenerate.py` collected from
`formal/tla/results.json` and `formal/tamarin/results/`. R1 is the admission bound
(calls never exceed $L$); R3 is the billing bound under provider faults.

In [ ]:
F = P["formal"]
print("TLC runs:", F["tlc_runs"], "| bounds:", F["tlc_bounds"], "| Tamarin limit:", F["tamarin_limit"])
out = []
for s in ORDER:
    t, m = F["tlc"][s], F["tamarin"][s]
    out.append(dict(implementation=LABEL[s],
                    tlc_R1="fails iff N>L" if t["r1_violated_exactly_when_N_gt_L"] else ("holds" if t["r1_holds_all"] else "see results"),
                    tlc_shortest_trace_steps=t["r1_shortest_counterexample_steps_L1"],
                    tlc_all_N_admitted_reachable=t["all_N_admitted_reachable"],
                    tlc_largest_run_states=t["max_distinct_states"],
                    tlc_R3=("n/a" if "r3_runs" not in t else
                            ("fails iff N>L" if t["r3_violated_exactly_when_N_gt_L"] else "holds")),
                    tamarin_R1=m["r1"], tamarin_steps=m["r1_steps"],
                    limit_reachable=t["limit_reachable"] and m["limit_reachable"]))
pd.DataFrame(out)

In [ ]:
# Fig. 8 and Table S14: the shortest TLC trace for count then insert, N = 2 requests, L = 1
pd.DataFrame(F["tlc_trace_count_then_insert_N2_L1"])

### Table X and Fig. 9: two off the shelf rate limiting libraries (archived round 6 arm)

SlowAPI and django-ratelimit, declared limit three, four Uvicorn workers, the barrier
protocol at the main sweep's six levels of $k$ (30 trials per cell), and two sequential
controls of 10 trials each: one keep alive connection, and a new connection per request.
Rerun with `study/run_round6.sh` (needs Redis and PostgreSQL); the event records are in
`results/r6/`.

In [ ]:
LB = P["libraries"]
rows = []
for cfg, x in LB["summary"].items():
    r = dict(configuration=cfg)
    for k in (1, 2, 4, 8, 16, 32):
        c = LB["cells"][f"{cfg}|{k}"]
        r[f"k={k} median/max"] = f"{c['median']:g}/{c['max']}"
    r["over (of 180)"] = x["over"]
    r["worst amplification"] = x["worst_amplification"]
    r["seq. one conn. max"] = LB["sequential"][f"{cfg}|sequential"]["max"]
    r["seq. new conn. max"] = LB["sequential"][f"{cfg}|sequential_new_conn"]["max"]
    rows.append(r)
display(pd.DataFrame(rows))
print("Django system checks:", LB["django_system_checks"])

### Figs. 7, 9 and 10, redrawn from the archived results

`study/make_figs.py` draws the TLC verdict map from `formal/tla/results.json`, the library
trials from `results/r6/libraries.csv`, and the comparison of every enforcement from
`results/paper_numbers.json`. Here they are written to a temporary folder, so the shipped
PDFs in `paper/` are not touched. Fig. 8 is TikZ, written by `study/paper_tables.py` into
`paper/gen/fig_trace.tex`.

In [ ]:
import tempfile
from IPython.display import Image, display as _show
import study.make_figs as MF
_tmp = pathlib.Path(tempfile.mkdtemp(prefix="otpfigs-"))
for fn, name in ((MF.formal_figure, "fig_formal"), (MF.libraries_figure, "fig_libraries"),
                 (MF.comparison_figure, "fig_comparison")):
    out = _tmp / f"{name}.png"
    fn(str(out))
    _show(Image(filename=str(out), width=480))

---
## Part D: every overrun, classified from its admission events

`results/overrun_classification.csv` has one row for each trial in any arm whose admissions
exceeded the declared limit. Round-4 rows carry the per-window admission counts from the
event records and the event file they came from; historical rows (rounds 1 to 3) carry no
per-window counts and the one one-step trial among them is marked indeterminate.

In [ ]:
cls = pd.read_csv(RESULTS / "overrun_classification.csv")
print(len(cls), "overruns recorded;", (cls["round"] == 4).sum(), "in round 4\n")
print(cls[cls["round"] == 4].groupby(["implementation", "classification"]).size().to_string())
print("\nround-4 policy violations by one-step implementations:", OV["round4_one_step_policy_violations"])
print("round-4 legitimate cross-window totals by one-step implementations:", OV["round4_one_step_legitimate_cross_window"])
print("\nhistorical:")
print(cls[cls["round"] != 4].groupby(["implementation", "classification"]).size().to_string())

### Checking one window assignment by hand

Take one legitimate cross-window total from the SQLite boundary test and reconstruct its
classification from the raw event records alone: count admissions per window identifier.

In [ ]:
import study.events as EV  # noqa: E402

row = cls[(cls["round"] == 4) & (cls.classification == "legitimate cross-window total")].iloc[0]
print(row.to_dict())
keys = ("arm", "backend", "strategy", "policy", "trial", "k", "limit", "window_s", "boundary")
tr = EV.trials(RESULTS / "r4" / (row.arm + ".events.jsonl"), keys)
evs = next(v for k, v in tr.items() if dict(zip(keys, k))["strategy"] == row.strategy
           and dict(zip(keys, k))["trial"] == row.trial)
from collections import Counter
per = Counter(e["window_id"] for e in evs if e["decision"] == "admit")
print("\nadmissions per window id, from the records:", dict(per))
print("largest count in any window:", max(per.values()), "<= limit", row.limit, "->",
      "no violation" if max(per.values()) <= row.limit else "violation")
print("provider calls recorded in the same records:", sum(1 for e in evs if e.get("message_id")))

---
## Study 1: what the ecosystem ships

The second measurement asks a prior question: before an application writes a limit of its
own, does the package it adopted already have one, on which operation, and how do we know?

The funnel from the complete package index down to the study sample:

1. the complete project index at the time of collection;
2. names matching one-time-password, two-factor-authentication and phone or SMS
   verification patterns;
3. project metadata consistent with code delivery;
4. downloaded and unpacked;
5. containing at least one **delivery site**, a function whose body both calls out to a
   messaging provider, email sender or HTTP endpoint *and* references the code value being
   delivered;
6. surviving manual adjudication.

Nineteen of the sixty-one auto-detected candidates were excluded by hand, and every exclusion
carries a recorded reason. The detector's recall was estimated by hand-labelling 60 of the 695
candidates it rejected; that estimate covers the detector stage only, within the 756
downloaded candidates.

In [ ]:
funnel = pd.DataFrame([
    ("Projects in the package index", S1["index_projects"]),
    ("Names matching OTP / 2FA / verification patterns", S1["name_matches"]),
    ("Metadata consistent with code delivery", S1["metadata_kept"]),
    ("Downloaded and unpacked", S1["downloaded_ok"]),
    ("Flagged by the delivery-site detector", S1["detector_flagged"]),
    ("Excluded on manual reading", S1["excluded_by_hand"]),
    ("Read by hand and retained", S1["retained"]),
], columns=["Stage", "Count"]).set_index("Stage")
print(funnel.to_string())
print()
print("send-side evidence levels:", S1["send_control_evidence_levels"])
print(f"bounded in the package's own code : {S1['send_control_in_package']}  ({S1['send_control_in_package_pct']}%, CI {S1['send_control_in_package_ci_pct']})")
print(f"delegated to a hosted provider    : {S1['send_control_delegated_provider']}")
print(f"no send-side bound                : {S1['send_control_no']}  ({S1['send_control_no_pct']}%)")
print(f"able to deliver by SMS            : {S1['sms_capable']}  (with a send-side bound: {S1['sms_capable_guarded']}, in package: {S1['sms_capable_guarded_in_package']})")
print()
print(f"detector validation: {S1['validation_misses']} misses in {S1['validation_sample']}; "
      f"recall within the screened set {S1['detector_recall_pct']}% (95% CI {S1['detector_recall_ci_pct']})")
print("scope:", S1["recall_scope"])

### Tables IV and V: send side against verification side controls, with the evidence for every label

The round-4 audit (`results/r4/classification_audit.csv`) records, for every package, the
evidence level and location behind the send-side label as well as the verification-side one.
Two labels changed in this round after a reviewer checked one against the source; both are
printed below with their evidence.

In [ ]:
ct = S1["crosstab"]
xt = pd.DataFrame(
    [[ct["send_yes_verify_yes"], ct["send_yes_verify_no"]],
     [ct["send_no_verify_yes"], ct["send_no_verify_no"]]],
    index=["send-side bound (in package or delegated)", "no send-side bound"],
    columns=["verification-side control", "none"])
xt["total"] = xt.sum(axis=1)
xt.loc["total"] = xt.sum()
display(xt)
print("both controls      :", ", ".join(S1["both_packages"]))
print("verification only  :", ", ".join(S1["verify_only_packages"]))
print("delegated          :", ", ".join(S1["delegated_packages"]))
print("helper not wired   :", ", ".join(S1["helper_not_wired_packages"]))
print("\nchanged in round 4:")
for k, v in S1["changed_in_round4"].items():
    print(f"  {k}: {v}")

In [ ]:
import csv as _csv  # noqa: E402

audit = list(_csv.DictReader(open(RESULTS / "r4" / "classification_audit.csv")))
print("send-side evidence for every package with a send-side bound:\n")
for r in audit:
    if r["send_control"] == "yes":
        print(f"  {r['package']:<28} {r['send_control_evidence_level']:<32} {r['send_control_evidence'][:110]}")
print("\nverification-side evidence for the verification-only packages:\n")
for r in audit:
    if r["send_control"] == "no" and r["verify_control"] == "yes":
        print(f"  {r['package']:<28} {r['verify_control_evidence'][:110]}")

### Re-running the survey from scratch

Study 1 re-downloads packages from the index, so its counts drift as the index changes. The
snapshot the paper used is in `results/survey_candidates.json`, and the per-package
adjudication is in `results/survey_final.csv`, which is why the cells above reproduce the
published numbers exactly.

```bash
python3 study/survey_discover.py    # full index -> name matches -> metadata filter
python3 study/survey_fetch2.py      # download and unpack the candidates
python3 study/survey_scope2.py      # AST pass: find delivery sites
python3 study/survey_gate.py        # look for a guard on each delivery site
python3 study/survey_final.py       # apply the recorded adjudication, write the CSV
python3 regenerate.py               # recompute every number in the paper
```

---
## Part R7. Round 7 (manuscript v5.0, service 2.2.0)

The round 7 arms answer the Computers & Security review. Every number below is read from
`results/paper_numbers.json` (key `r7`), which `regenerate.py` computes from the records in
`results/r7/` and the checker outputs in `formal/`; the commands that produced the records are
in `study/run_round7.sh` and `study/run_round7_resume.sh`. Section and table numbers refer to
manuscript v5.0.

In [ ]:
import json, pandas as pd
P = json.load(open("results/paper_numbers.json"))
R7 = P["r7"]
pk = pd.DataFrame([{"package": d["package"], "version": d["version"], "state": d["backend"],
                    "limit": d["declared_limit"], "trials": d["trials"],
                    "max simultaneous": d["concurrent_max"],
                    "over (k > L)": f'{d.get("concurrent_over", "-")}/{d.get("concurrent_trials_k_gt_L", "-")}',
                    "sequential max": d.get("sequential_max", "-")} for d in R7["packages"].values()])
pk   # Table pkgexec: six surveyed packages executed on their own delivery paths

In [ ]:
lib = pd.DataFrame([dict(config=c, **{k: v for k, v in d.items()}) for c, d in R7["libraries7"]["summary"].items()])
lib  # the seven round 7 library configurations (Table libraries)

In [ ]:
print("independent runs: one step violations", R7["indep"]["new_runs_one_step_violations"],
      "in", R7["indep"]["new_runs_one_step_trials"], "new trials")
print("independent provider:", {b: {k: v for k, v in d.items() if k != "by_strategy"} for b, d in R7["provider"].items()})
pd.DataFrame(R7["timing_events"]).T   # the time and fault arm, recomputed request by request (Table timing)

In [ ]:
pd.DataFrame(R7["load"]).T   # sustained open loop load (Table load)

In [ ]:
F = R7["formal"]
print("timed TLC runs:", F.get("tlc_timed_runs"), " largest state space:", F.get("tlc_timed_max_states"))
pd.DataFrame([{"run": k, "verdict": "holds" if v["holds_all"] else "violated"} for k, v in F.get("tlc_timed", {}).items()])

---
## What this shows, and what it does not

**Shows.** On an ordinary Python stack, three of six natural ways to write the same quota
check admit up to 10.7x the declared allowance when requests overlap, while reporting success
for every request; the overrun does not depend on the provider being slow; the same requests
issued one at a time make all six look correct; the admission events show which requests read
stale state and which counter increments were lost; the three one-step implementations
violated their declared policy in no event-logged trial, including the ones that straddle a
window boundary on purpose; and a surveyed package executed as shipped behaves like the
two-step fixtures. Separately, ten of 42 sampled packages bound sending in their own code,
three delegate it, twenty-nine do not, and ten bound only the operation that does not spend
money.

**Does not show.** One language ecosystem, and the packages one search discovers (every filter stage is validated in round 7).
Two datastores on one host. Fixtures that are structural analogues of surveyed code, not
copies (six surveyed packages are executed as shipped in round 7, Part R7). Application-observed instants, not database commit
order, for the timing comparisons. Nothing about caller-level guards the integrating
application may add. And no real billing: every send was recorded by a local stub.

## If you are fixing this in your own code

1. **Make the admission decision one operation the datastore serialises**, and have the
   datastore tell you the window it counted in.
2. **State the policy precisely, and test against that event.** A fixed-window limiter
   legitimately admits up to 2L across a boundary; a sliding-window limiter does not; and a
   test that assigns windows from a later event, such as the provider call, will call a
   correct implementation wrong.
3. **Claim the allowance before the send, not after.**
4. **Enumerate the send routes.**
5. **Anchor the quota key server-side.**

---
### Citation and license

```bibtex
@article{ahmad2026otpsend,
  author  = {Ahmad, Naveed},
  title   = {Send Side Quota Enforcement in One Time Password Software:
             A Package Survey and a Measurement Under Concurrency},
  journal = {Computers \& Security},
  year    = {2026},
  note    = {Under review; artifact v5.0-r17}
}
```

The code in this repository is released under the MIT license. The package metadata in
`results/` was collected from the public package index and remains subject to the licenses
of the projects it describes.